# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant


## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_obj = dataset.metadata

print(f"{getattr(metadata_obj, 'name', 'Dataset')}: {getattr(metadata_obj, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below we list all available record sets and their corresponding fields using their `@id`s. These will be used for downstream data loading and analysis.

In [ ]:
# List all record sets and their fields by @id

record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set @id: {rs.id}")
    print(f"  name: {rs.name if hasattr(rs, 'name') else ''}")
    print(f"  Fields and Columns:")
    for field in rs.fields:
        print(f"    Field @id: {field.id}, name: {field.name if hasattr(field, 'name') else ''}")
        if hasattr(field, 'column') and getattr(field, 'column', None):
            # In Croissant, field.column can be str or list
            columns = field.column if isinstance(field.column, (list, tuple)) else [field.column]
            for col in columns:
                print(f"      Column @id: {col.id if hasattr(col, 'id') else col}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

You can select the main survey record set of interest by its `@id`. Here, we show how to load all record sets into pandas DataFrames using their `@id`s for later analysis.

In [ ]:
# Gather all record set @id strings
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set '{record_set_id}' with {len(dataframes[record_set_id])} records and columns:")
        print(dataframes[record_set_id].columns.tolist())
        print(dataframes[record_set_id].head(2))
    else:
        print(f"Record set '{record_set_id}' is present but has no records.")
    print("-"*60)

# Choose the main survey data by inspecting IDs and columns
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
# You can override main_record_set_id here after inspecting outputs above, if needed.

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, and grouping data by key attributes to prepare for analysis.

Specify the `@id` of the record set and field you wish to analyze (as determined above).

In [ ]:
# Example: Choose the first available numeric field (e.g., PHQ-9 score) for demonstration
rs_df = None
if main_record_set_id and main_record_set_id in dataframes:
    rs_df = dataframes[main_record_set_id]

    print(f"Fields in main record set ({main_record_set_id}):\n", rs_df.columns.tolist())

    # Example field IDs -- change as appropriate to match your data
    # Let's try to find a field related to PHQ, GAD, or PSQ scores
    candidates = [col for col in rs_df.columns if 'phq' in col.lower() or 'gad' in col.lower() or 'psq' in col.lower() or 'score' in col.lower()]
    if candidates:
        numeric_field = candidates[0]
        print(f"Using numeric field: {numeric_field}")
    else:
        numeric_field = rs_df.columns[0]  # fallback: first column
        print(f"Falling back to: {numeric_field}")

    threshold = rs_df[numeric_field].mean() if pd.api.types.is_numeric_dtype(rs_df[numeric_field]) else 10
    print(f"Using threshold: {threshold}")
    
    filtered_df = rs_df
    if pd.api.types.is_numeric_dtype(rs_df[numeric_field]):
        filtered_df = rs_df[rs_df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (size: {len(filtered_df)}):")
    else:
        print("Chosen field is not numeric; skipping filtering.")

    print(filtered_df.head())
    
    # Normalization
    if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Group by a demographic field, e.g. 'gender' or 'age_group' if available
    group_candidates = [col for col in rs_df.columns if 'gender' in col.lower() or 'age' in col.lower() or 'village' in col.lower() or 'education' in col.lower()]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"Grouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())
    else:
        print("No groupable field found for aggregation.")
else:
    print("No valid DataFrame loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We provide example plots for a chosen numeric field (e.g., PHQ-9, GAD-7, or PSQ score) and its distribution across a demographic attribute.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if rs_df is not None and pd.api.types.is_numeric_dtype(rs_df[numeric_field]):
    plt.figure(figsize=(8,4))
    sns.histplot(rs_df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    # Boxplot by group field if available
    if group_candidates:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=rs_df[group_candidates[0]], y=rs_df[numeric_field])
        plt.title(f"{numeric_field} by {group_candidates[0]}")
        plt.xlabel(group_candidates[0])
        plt.ylabel(numeric_field)
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains mental health indicators and demographic information collected in Kilifi County, Kenya.
- Using `mlcroissant`, we loaded record sets, inspected the field structure via their `@id`s, performed sample filtering, normalization, aggregation, and visualized distributions.
- This notebook provides a blueprint for further in-depth analysis using the FAIR² dataset. Refer to the Croissant schema documentation and the field `@id`s for precise data mapping in your projects.
